# Tiền xử lý dữ liệu
Chuẩn bị đặc trưng, target hồi quy và hai bộ tiền xử lý để ghép vào Pipeline. Mức chi phí được suy ra từ số tiền dự đoán bằng ngưỡng cố định, không huấn luyện classifier riêng. Chỉ đọc Train; chưa fit encoding/scaling.

In [1]:
from pathlib import Path
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "src":
    ROOT = ROOT.parent

train = pd.read_csv(ROOT / "data" / "medical_insurance_train.csv")
print(f"Train: {len(train):,} dòng, {train.shape[1]} cột")

Train: 1,069 dòng, 7 cột


## 1. Tách đặc trưng và target Regression
X gồm 6 thuộc tính đầu vào; `charges` chỉ dùng làm target, không đưa vào bộ tiền xử lý đặc trưng.

In [2]:
numeric = ["age", "bmi", "children"]
categorical = ["sex", "smoker", "region"]

X_train = train[numeric + categorical].copy()
y_train_reg = train["charges"].copy()

print("X_train:", X_train.shape)
print("y_train_reg:", y_train_reg.shape)
display(X_train.head())

X_train: (1069, 6)
y_train_reg: (1069,)


,age,bmi,children,sex,smoker,region
0,23,24.510,0,male,no,northeast
1,21,25.745,2,male,no,northeast
2,52,37.525,2,female,no,northwest
3,63,41.470,0,male,no,southeast
4,47,26.600,2,female,no,northeast


## 2. Thiết lập encoding và scaling
- One-Hot Encoding cho `sex`, `smoker`, `region`; nhãn chưa gặp khi fit được xử lý bằng `handle_unknown="ignore"`.
- Bộ **scaled**: StandardScaler cho cột số, phù hợp mô hình nhạy với thang đo như Ridge hoặc SVR.
- Bộ **tree**: giữ nguyên cột số, dành cho các mô hình cây như Random Forest/XGBoost.

Chỉ khởi tạo, chưa gọi `fit` hay `fit_transform`. Khi thử mô hình, ghép bộ phù hợp vào Pipeline để mỗi fold CV tự fit từ phần train của fold. Dùng one-hot đầy đủ; nếu thử Linear Regression không điều chuẩn, cần xét tính phụ thuộc giữa các cột dummy khi diễn giải hệ số.

In [3]:
preprocessor_scaled = ColumnTransformer([
    ("numeric", StandardScaler(), numeric),
    ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
])

preprocessor_tree = ColumnTransformer([
    ("numeric", "passthrough", numeric),
    ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
])

print("Đã tạo hai bộ tiền xử lý: scaled và tree (chưa fit).")

Đã tạo hai bộ tiền xử lý: scaled và tree (chưa fit).


## 3. Xác định ngưỡng phân mức chi phí
Giữ quy ước hiện tại: phân vị **1/3 và 2/3 của charges thực tế trên Train**. Đây là ba mức tương đối trong dữ liệu, không phải ngưỡng y tế. Chưa đổi sang Q1/Q3.

Dùng cùng ngưỡng cho chi phí thực tế và dự đoán khi đánh giá mức chi phí trên Test. Không tính lại ngưỡng từ Test; không dùng mức chi phí làm đầu vào mô hình.

In [4]:
cost_quantiles = [1 / 3, 2 / 3]
lower, upper = y_train_reg.quantile(cost_quantiles).tolist()
cost_labels = ["Thấp", "Trung bình", "Cao"]

# Chỉ minh họa phân bố mức chi phí thực tế, không tạo target classifier.
train_cost_levels = pd.cut(
    y_train_reg,
    bins=[-float("inf"), lower, upper, float("inf")],
    labels=cost_labels,
    right=True,
)
print(f"Thấp: charges <= {lower:,.4f} USD")
print(f"Trung bình: {lower:,.4f} < charges <= {upper:,.4f} USD")
print(f"Cao: charges > {upper:,.4f} USD")
display(train_cost_levels.value_counts(sort=False).rename("Số bản ghi Train").to_frame())

Thấp: charges <= 6,250.4350 USD
Trung bình: 6,250.4350 < charges <= 12,730.9996 USD
Cao: charges > 12,730.9996 USD


,Số bản ghi Train
charges,
Thấp,357
Trung bình,356
Cao,356


## 4. Lưu cấu hình dùng chung
Lưu hai bộ tiền xử lý **chưa fit**, nhóm đặc trưng và quy tắc phân mức trong một file Joblib. File này chưa phải mô hình dự đoán. Không xuất CSV đã scaling toàn bộ Train trước CV.

In [5]:
MODELS = ROOT / "models"
MODELS.mkdir(parents=True, exist_ok=True)
OUTPUT = MODELS / "preprocessing_setup.joblib"

setup = {
    "numeric_features": numeric,
    "categorical_features": categorical,
    "target": "charges",
    "scaled": preprocessor_scaled,
    "tree": preprocessor_tree,
    "cost_quantiles": cost_quantiles,
    "cost_thresholds": [lower, upper],
    "cost_labels": cost_labels,
}
joblib.dump(setup, OUTPUT)
print(f"Đã lưu: {OUTPUT.relative_to(ROOT)}")

Đã lưu: models/preprocessing_setup.joblib


**Bước tiếp theo:** `regression_experiments.ipynb` tải cấu hình, ghép bản sao bộ tiền xử lý với từng mô hình bằng Pipeline và so sánh bằng CV trên Train. Ví dụ cấu trúc, với `model` là mô hình đang thử:

```python
from sklearn.base import clone
from sklearn.pipeline import Pipeline

setup = joblib.load(ROOT / "models" / "preprocessing_setup.joblib")
pipeline = Pipeline([
    ("preprocessing", clone(setup["scaled"])),  # Hoặc setup["tree"]
    ("model", model),
])
```

Sau khi chọn mô hình và tham số bằng CV, `train_final.ipynb` fit Pipeline trên toàn bộ Train rồi đánh giá Test. Áp dụng `cost_thresholds` cho cả chi phí thực tế và dự đoán để đánh giá phân mức. Lưu Pipeline đã huấn luyện cùng ngưỡng phục vụ web; không huấn luyện mô hình Classification riêng.